In [0]:
%run ./00_config

In [0]:
df_silver = spark.read.format("delta").load(SILVER_PATH)

df_silver.show()

In [0]:
df_gold_candidate = df_silver.select(*GOLD_COLUMNS)

In [0]:
from pyspark.sql import functions as F

total_rows = df_gold_candidate.count()
ca_rows = df_gold_candidate.filter(F.col("state")=="CA").count()

print(f"Total rows: {total_rows}")
print(f"CA rows : {ca_rows}")

if total_rows == 0:
    raise RuntimeError("Gold Publish aborted: batch is completely empty")

if ca_rows == 0:
    raise RuntimeError("Gold Publish aborted: batch is missing CA rows - This violates the expected baseline (CA should always have active chapters)")

print("BATCH VALIDATION PASSED - proceeding with GOLD publish")


In [0]:
# df_gold_check = spark.read.format("delta").load(GOLD_PATH)
# df_gold_check.show()
# print(f"Gold row count: {df_gold_check.count()}")

In [0]:
from delta.tables import DeltaTable

if DeltaTable.isDeltaTable(spark, GOLD_PATH):
    gold_table = DeltaTable.forPath(spark, GOLD_PATH)

    gold_table.alias("target").merge(
        df_gold_candidate.alias("source"),
        "target.chapter_id = source.chapter_id"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

    print("MERGE complete — Gold table updated.")

else:
    df_gold_candidate.write.format("delta").save(GOLD_PATH)
    print("Gold table does not exist - initial gold table created")